In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

returns = pd.read_csv("india_equity_returns.csv", index_col="Date", parse_dates=True)

mean_returns = returns.mean()
cov_matrix = returns.cov()

n = len(mean_returns)

weights_equal = np.repeat(1/n, n)

ret_equal = np.sum(mean_returns * weights_equal)
vol_equal = np.sqrt(weights_equal.T @ cov_matrix @ weights_equal)

def portfolio_perf(w):
    r = np.sum(mean_returns * w)
    v = np.sqrt(w.T @ cov_matrix @ w)
    return r, v

def neg_sharpe(w, rf=0):
    r, v = portfolio_perf(w)
    return -(r - rf) / v

constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0,1) for _ in range(n))
init = np.repeat(1/n, n)

opt = minimize(neg_sharpe, init, method='SLSQP', bounds=bounds, constraints=constraints)

weights_opt = opt.x
ret_opt, vol_opt = portfolio_perf(weights_opt)

out = pd.DataFrame({
    "Asset": mean_returns.index,
    "Equal_Weight": weights_equal,
    "Optimized_Weight": weights_opt
})

out.to_csv("portfolio_weights.csv")

summary = pd.DataFrame({
    "Portfolio":["Equal Weight","Optimized"],
    "Return":[ret_equal, ret_opt],
    "Volatility":[vol_equal, vol_opt]
})

summary.to_csv("portfolio_summary.csv")

out

,Asset,Equal_Weight,Optimized_Weight
0,RELI,0.111111,1.639856e-18
1,HDBK,0.111111,0.000000e+00
2,EICH,0.111111,1.038205e-01
3,HLL,0.111111,1.233280e-18
4,BRTI,0.111111,5.216432e-01
5,LART,0.111111,0.000000e+00
6,SUN,0.111111,1.602255e-01
7,TITN,0.111111,1.723565e-01
8,TISC,0.111111,4.195437e-02
